In [ ]:
import sys; sys.path.insert(0, '../..')
from paths import ROOT, DATA, STEPS, PREPARES, FIGURES, OUTPUT, CODE


In [3]:
import h5py
import pickle
import scipy
import numpy as np
import seaborn as sns
sns.set()
# sns.set_style("whitegrid", {'axes.grid' : False})
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from os.path import join as pjoin
import scipy.io as sio
from scipy import stats
import scipy.signal as sp
from scipy import signal
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from statannot import add_stat_annotation
from skimage import io
from skimage.feature import hog
from joblib import Parallel, delayed
import cv2
import glob
import time
from skimage.registration import phase_cross_correlation
from scipy.ndimage import fourier_shift
from joblib import Parallel, delayed
from skimage.feature import hog
import sys
sys.path.insert(1, str(CODE / 'make_0_prepare' / 'common-functions'))
from harness import *


from IPython.display import display, clear_output, HTML, Image
from matplotlib import animation, rc

from skimage.measure import regionprops

from matplotlib import cm as mpl_cm
from matplotlib import (colors,
                        lines,
                        transforms,
                        )

from f2_utils import *
import time_in_each_roi

# this is copied from /sensory-mapping

In [ ]:
step_0_root = str(STEPS / "step_0") + "/"
step_1_root = str(STEPS / "step_1") + "/"
step_2_root = str(STEPS / "step_2") + "/"
step_3_root = str(STEPS / "step_3") + "/"

fetch_path_S1 = str(PREPARES / "index-S1") + "/"
fetch_path_M2 = str(PREPARES / "index-M2") + "/"
fetch_path_M1 = str(FIGURES / "figure5") + "/"
fetch_path_extra = str(PREPARES / "index-extra") + "/"
figure_config_root = str(PREPARES / "config") + "/"


In [5]:
def show_map(data,caxis,if_return_map=False,cmap='rocket'):
    ss_idx = IDX_sorted.shape;
    IDX_reshape = IDX_sorted.reshape((-1,1))
    tmp = np.empty((ss_idx[0]*ss_idx[1],1))
    tmp[:] = np.nan
    std_map = tmp
    ncomps=500
    for i_region in range(ncomps):
        indice = IDX_reshape == i_region+1
        std_map[indice] =  data[i_region]
    # reshape back
    std_map = np.reshape(std_map,(ss_idx[0],ss_idx[1]))

    if if_return_map ==True:
        return std_map
    else:
        plt.imshow(std_map,vmin=caxis[0],vmax=caxis[1],cmap=cmap)
        plt.axis('off')
        plt.colorbar()
def save_model(model,pkl_filename):
    with open(pkl_filename, 'wb') as file:
        pickle.dump(model, file)

def load_model(pkl_filename):
    with open(pkl_filename, 'rb') as file:
        pickle_model = pickle.load(file)
    return pickle_model


In [ ]:
mouse='cm241'
# with h5py.File(step_0_root+mouse+'/'+'idx_sorted_500_'+mouse+'.mat','r') as f_idx:

#     IDX_sorted = np.array(f_idx['IDX_sorted']).T
#     IDX_sorted = IDX_sorted.astype(float)
#     IDX_sorted[IDX_sorted==0]='nan'
#     num_regions = int(np.nanmax(IDX_sorted.reshape((-1,1))))
#     props = regionprops(IDX_sorted.astype(int))
#     node_coords = [props[i].centroid for i in range(len(props))]
#     node_coords = np.asarray(node_coords, dtype=np.float32)
# plt.imshow(IDX_sorted)

In [ ]:
load_path = str(PREPARES / "registration") + "/"
dict_transform = load_model(load_path+'dict_transform')
M=dict_transform[mouse]




# 1. spontaneous running

In [8]:
bodyparts = ['RW','LW','tail','RFP','LFP','nose','LHP','RHP','mouth','RT','LT']


day4_dict = {
     'runD':"boop_on = 'RW';stim_frames_raw="+"[1510, 3308, 5100, 8695, 10495, 12294, 14090, 15888, 17683, 19481, 21279, 23075, 24872, 26670, 28466, 30263, 32060, 33857];",
     'runE':"boop_on = 'LW';stim_frames_raw="+"[1510, 3308, 5100, 8695, 10495, 12294, 14090, 15888, 17683, 19481, 21279, 23075, 24872, 26670, 28466, 30263, 32060, 33857];",

     'runF':"boop_on = 'tail';stim_frames_raw="+"[1543 ,3320,5101,6897,8700,10493,12278,14089,15860];",
    'runG':"boop_on = 'RFP';stim_frames_raw="+"[1546, 3352,5120,6912,8702,10495,12311,14174];",
    'runH':"boop_on = 'LFP';stim_frames_raw="+"[1581, 3310,5010,6935,8735,10494,12351,14077,15894];",
    'runI':"boop_on = 'nose';stim_frames_raw="+"[1526, 3318,5106,6896,8662,10463,12268,14087,15908];",
    'runJ':"boop_on = 'LHP';stim_frames_raw="+"[1516, 3277,5080,6880,8690,10440,12235,14083,15880];",
    'runK':"boop_on = 'RHP';stim_frames_raw="+"[1516, 3277,5080,6880,8690,10440,12235,14083,15880];",
    'runL':"boop_on = 'mouth';stim_frames_raw="+"[3278 ,5099,6924,8714,10479,12296,14082,15880];",
    'runM':"boop_on = 'RT';stim_frames_raw="+"[1526, 3318,5106,6896,8662,10463,12268,14087,15908];",
    'runN':"boop_on = 'LT';stim_frames_raw="+"[1540, 3290, 5080, 6890, 8703, 10490, 12270, 14080, 15850];"

}

In [ ]:

f4 = [f for f in glob.glob(step_1_root+mouse+"/[4]/run[D-N]*.mat", recursive=True)]

file_spont = f4
file_spont

In [10]:
sns.set()
import warnings
warnings.filterwarnings('ignore')
###################################################################### M setting

T_PRE = 100
T_POST = 200
T = T_PRE+T_POST
num_regions = 500

M_jrgeco = {key:np.empty((256,256,T,0)) for key in bodyparts}
M_day_label = {key:np.empty((0)) for key in bodyparts}

for i_file,fname in enumerate(file_spont): 
    ################################### identity 
    mouse = fname.split('step_1/')[1][0:5]
    day = fname.split(mouse)[1][1:2]
    mouse_day = mouse+'_'+day
    run_name = fname.split('_stim_1')[0].split('/'+day+'/')[1]
    run = run_name
    if day=='4':
        exec(day4_dict[run]);print(day4_dict[run][:15])
        total_frames = 12000
    stim_frames = np.round(np.array(stim_frames_raw)/3,decimals=-1).astype('int')
    
    
    start_stop_ind = pd.DataFrame(stim_frames)
    start_stop_ind = start_stop_ind[start_stop_ind[0]<total_frames-T_POST]
    for i in range(len(start_stop_ind)):
        
        cut_range = range(start_stop_ind.iloc[i,0]-T_PRE,start_stop_ind.iloc[i,0]+T_POST)

        with h5py.File(fname, "r") as f_H:
            m = f_H['m']
            C = np.array(f_H['C']['jrgeco'])
            S = np.array(f_H['S']['jrgeco'])[cut_range,:]
            # cut [-60,60] frames slice
            mask = np.array(f_H['mask'])
            # nanidx = np.squeeze(~np.isnan(mask.reshape((1,-1))))
            nanidx = np.squeeze(m['nanidx'])==1
            CS = np.dot(S,C).transpose()
            jrgeco = np.zeros((len(nanidx),CS.shape[-1]))
            jrgeco[:] = np.nan
            jrgeco[nanidx==True,:] = CS
            jrgeco = jrgeco.reshape((256,256,-1)).transpose((1,0,2))

        jrgeco_dsl = jrgeco-np.mean(jrgeco[:,:,:80],axis=2)[:,:,np.newaxis]
        M_jrgeco[boop_on] = np.append(M_jrgeco[boop_on],jrgeco_dsl[:,:,:,np.newaxis],axis=3)
        M_day_label[boop_on] = np.append(M_day_label[boop_on],mouse_day+'_'+run)


boop_on = 'RW';
boop_on = 'LFP'
boop_on = 'LW';
boop_on = 'tail
boop_on = 'nose
boop_on = 'LHP'
boop_on = 'RT';
boop_on = 'LT';
boop_on = 'mout
boop_on = 'RFP'
boop_on = 'RHP'


In [11]:
from skimage import measure


In [ ]:
sns.set_style("whitegrid", {'axes.grid' : False})
avg_brain_maps = {key:[] for key in bodyparts}
contours = {key:[] for key in bodyparts}
clean_contours = {key:[] for key in bodyparts}
brightest_ind = {key:[] for key in bodyparts}
vmax_dict = {key:[] for key in bodyparts}
t_range_dict = {key:[] for key in bodyparts}
homunculus_rois = {key:[] for key in bodyparts}


In [ ]:
from sklearn.preprocessing import minmax_scale


In [ ]:
def build_bodypart_roi(
    name,
    t_range,
    vmax,
    bw_threshold,
    reject_contour,
    focus_range=(0.1, 1),
    title=None,
    show_bw_preview=True,
    abs_signal=False,
    mask_crop=None,
    source_key=None,
    stat_key=None,
    vmin=0,
    select_bouts=None,
    precomputed_brain_map=None,
    skip_clean=False,
):
    # stat_key lets vmax_dict/t_range_dict be written under a different key than `name`
    # -- used when deriving one body part's ROI from another's brain_map.
    t_range_dict[stat_key or name] = t_range
    if precomputed_brain_map is not None:
        brain_map = precomputed_brain_map
    else:
        signal = M_jrgeco[source_key or name]
        if select_bouts is not None:
            signal = signal[:, :, :, select_bouts]
        if abs_signal:
            signal = np.abs(signal)
        brain_map = np.mean(np.mean(signal, axis=-1)[:, :, t_range], axis=-1)
        brain_map = cv2.warpAffine(brain_map, M[0:2, :], (256, 256))
    vmax_dict[stat_key or name] = vmax

    plt.figure(figsize=(8, 3), dpi=300)
    plt.subplot(121)
    plt.imshow(brain_map, vmin=vmin, vmax=vmax, cmap='coolwarm')
    plt.colorbar()
    plt.xticks([]); plt.yticks([])
    plt.title(title or name)

    plt.subplot(122)
    bw = brain_map > bw_threshold
    if show_bw_preview:
        plt.imshow(bw)
    filled_bw = scipy.ndimage.morphology.binary_fill_holes(bw)
    contours[name] = measure.find_contours(filled_bw, 0.8)

    if skip_clean:
        # LT/RT's original cells computed contours but never filtered/plotted them --
        # clean_contours[name] stays at its initialized [] and homunculus_rois[name]
        # is never written. Preserved here rather than silently filling in the gap.
        avg_brain_maps[name] = brain_map
        return brain_map

    if mask_crop is not None:
        filled_bw = mask_crop(filled_bw)

    filled_bw_nan = filled_bw.astype('float')
    filled_bw_nan[filled_bw_nan == 0] = np.nan
    focus_map = np.multiply(filled_bw_nan, brain_map)
    shape = focus_map.shape
    focus_map_scaled = minmax_scale(focus_map.ravel(), feature_range=focus_range).reshape(shape)
    plt.imshow(focus_map_scaled)
    homunculus_rois[name] = focus_map_scaled

    clean_contours[name] = [c for c in contours[name] if not reject_contour(c)]
    for c in clean_contours[name]:
        plt.plot(c[:, 1], c[:, 0], linewidth=2)

    avg_brain_maps[name] = brain_map
    return brain_map


def _reject_default(contour, min_len=20, row_cutoff=230):
    return len(contour) < min_len or np.any(contour[:, 0] > row_cutoff)


In [ ]:
build_bodypart_roi(
    'LFP', range(T_PRE, T_PRE + 60), vmax=0.07, bw_threshold=0.045,
    reject_contour=_reject_default,
)


In [ ]:
build_bodypart_roi(
    'RFP', range(T_PRE, T_PRE + 40), vmax=0.07, bw_threshold=0.045,
    reject_contour=_reject_default,
)


In [ ]:
build_bodypart_roi(
    'LHP', range(T_PRE, T_PRE + 40), vmax=0.07, bw_threshold=0.025,
    reject_contour=_reject_default,
)


In [ ]:
build_bodypart_roi(
    'RHP', range(T_PRE, T_PRE + 50), vmax=0.06, bw_threshold=0.03,
    reject_contour=_reject_default,
)


In [ ]:
build_bodypart_roi(
    'mouth', range(T_PRE, T_PRE + 100), vmax=0.07, bw_threshold=0.045,
    select_bouts=[0, 1, 2, 5, 6], reject_contour=_reject_default,
)


In [ ]:
build_bodypart_roi(
    'nose', range(T_PRE, T_PRE + 30), vmax=0.07, bw_threshold=0.055,
    reject_contour=_reject_default,
)


In [ ]:
build_bodypart_roi(
    'tail', range(T_PRE + 10, T_PRE + 50), vmax=0.07, bw_threshold=0.05,
    reject_contour=_reject_default,
)


In [ ]:
build_bodypart_roi(
    'LW', range(T_PRE, T_PRE + 40), vmax=0.13, bw_threshold=0.11,
    reject_contour=_reject_default,
)


In [ ]:
build_bodypart_roi(
    'RW', range(T_PRE + 20, T_PRE + 80), vmax=0.1, bw_threshold=0.09,
    reject_contour=lambda c: (
        len(c) < 20 or np.any(c[:, 0] > 200) or np.any(c[:, 1] > 200)
    ),
)


In [ ]:
build_bodypart_roi(
    'LT', range(T_PRE, T_PRE + 40), vmax=0.06, bw_threshold=0.04,
    reject_contour=_reject_default, skip_clean=True,
)


In [ ]:
build_bodypart_roi(
    'RT', range(T_PRE, T_PRE + 40), vmax=0.04, bw_threshold=0.03,
    reject_contour=_reject_default, skip_clean=True,
)


In [ ]:
save_path = str(PREPARES / "sensory-boop-test") + "/"
save_model(clean_contours, save_path + 'clean_contours_' + mouse)
